# Análisis del Motor de Búsqueda Semántica

Antes de ejecutar este notebook asegúrate de haber corrido `indexar.py` para poblar ChromaDB.

In [ ]:
from buscar import buscar
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os
import chromadb
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
print('Librerías cargadas correctamente')

## Parte 1: Queries de prueba y sus resultados

In [ ]:
queries = [
    '¿cómo hacer una API en Python?',
    'diferencias entre frameworks de frontend',
    'cómo funciona la autenticación en aplicaciones web',
    'herramientas para trabajar con modelos de lenguaje',
]

todos_resultados = {q: buscar(q, n_resultados=3) for q in queries}

for q, resultados in todos_resultados.items():
    print(f'\nQuery: {q!r}')
    for r in resultados:
        print(f'  [{r["score"]:.4f}] {r["titulo"]}')

## Parte 2: Tabla comparativa

In [ ]:
filas = []
for q, resultados in todos_resultados.items():
    mejor = resultados[0]
    filas.append({
        'Query': q,
        'Mejor resultado': mejor['titulo'],
        'Score': mejor['score'],
        'Contenido (resumen)': mejor['contenido'][:60] + '...'
    })

df_tabla = pd.DataFrame(filas)
df_tabla.style.background_gradient(subset=['Score'], cmap='RdYlGn').format({'Score': '{:.4f}'})

## Bonus: Mapa de calor de similitud entre todos los artículos

In [ ]:
MODELO_EMBEDDING = 'text-embedding-3-small'

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
chroma = chromadb.PersistentClient(path='./chroma_db')
coleccion = chroma.get_or_create_collection(name='articulos_tech')

datos = coleccion.get(include=['embeddings', 'metadatas'])
titulos = [m['titulo'] for m in datos['metadatas']]
embeddings = np.array(datos['embeddings'])

# Similitud coseno entre cada par de artículos
normas = np.linalg.norm(embeddings, axis=1, keepdims=True)
embeddings_norm = embeddings / normas
similitud = embeddings_norm @ embeddings_norm.T

df_similitud = pd.DataFrame(similitud, index=titulos, columns=titulos)

plt.figure(figsize=(10, 8))
sns.heatmap(
    df_similitud,
    annot=True,
    fmt='.2f',
    cmap='YlOrRd',
    vmin=0,
    vmax=1,
    linewidths=0.5,
    xticklabels=[t[:20] for t in titulos],
    yticklabels=[t[:20] for t in titulos],
)
plt.title('Similitud coseno entre artículos', fontsize=14)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.show()

## Top 3 resultados por query (detalle completo)

In [ ]:
filas_detalle = []
for q, resultados in todos_resultados.items():
    for pos, r in enumerate(resultados, 1):
        filas_detalle.append({
            'Query': q[:40] + ('...' if len(q) > 40 else ''),
            'Posición': pos,
            'Título': r['titulo'],
            'Score': r['score'],
        })

df_detalle = pd.DataFrame(filas_detalle)
df_detalle.style.background_gradient(subset=['Score'], cmap='Blues').format({'Score': '{:.4f}'})